# XQuality — Resume prefix stop-after-layer LLM finalization from Layer 6

This notebook resumes the **prefix stop-after-layer finalization** experiment from Layer 6 onward.

Stopping after Layer X means the finalizer receives the whole prefix state available up to X, i.e. Layer 0..X. For Layer 6, it uses all information saved in the Layer 6 state, which already contains the previous layer results.

It calls OpenRouter/LiteLLM in parallel with batched JSONL output and writes completion markers so the evaluation notebook can distinguish complete from interrupted outputs.


In [ ]:

from __future__ import annotations

import os
import sys
import json
import re
import time
import math
import shutil
from pathlib import Path
from typing import Any
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 140)


In [ ]:

# -------------------------
# Repository / experiment config
# -------------------------
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src' / 'neoolaf').exists() or (p / 'pyproject.toml').exists():
            return p
    for p in [start, *start.parents]:
        if (p / 'examples' / 'XQualityMachine32').exists():
            return p
    raise RuntimeError(f'Could not find NeoOLAF repo root from {start}')

ROOT = find_repo_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Same model/env as the rest of your experiments.
MODEL = 'openrouter/openai/gpt-oss-20b'
TEMPERATURE = 0.0
MAX_TOKENS = 4500
MAX_WORKERS = 4
MAX_RETRIES = 2
RETRY_SLEEP_SECONDS = 2

# Batching keeps calls much lower than one call per unit.
BATCH_SIZE_UNITS = 10
MAX_BATCH_CHARS = 12000

START_STOP_INDEX = 6
END_STOP_INDEX = None  # None means go to the last discovered layer
FORCE_RERUN = False
DELETE_INCOMPLETE_START_FOLDER = True

PROFILE_NAME = 'xquality_loose'
RUNS_ROOT = ROOT / 'examples' / 'XQualityMachine32' / 'runs'
OUTPUT_ROOT = RUNS_ROOT / 'prefix_stop_after_layer_llm_finalization_exact_eval'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('RUNS_ROOT =', RUNS_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('MODEL =', MODEL)
print('MAX_WORKERS =', MAX_WORKERS)


In [ ]:

# Optional: load .env for OPENROUTER_API_KEY / LITELLM settings.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
    load_dotenv(Path.cwd() / '.env')
except Exception:
    pass

try:
    import litellm
    from litellm import completion
except Exception as e:
    raise RuntimeError('This notebook requires litellm in the current NeoOLAF environment.') from e

print('OPENROUTER_API_KEY set:', bool(os.environ.get('OPENROUTER_API_KEY')))


In [ ]:

# -------------------------
# Discover saved layer states
# -------------------------
LAYER_NAMES = [
    'layer00_preprocessing',
    'layer01_linguistic_expression_extraction',
    'layer02_candidate_enrichment',
    'layer03_candidate_typing_resolution',
    'layer04_candidate_relation_extraction',
    'layer05_candidate_triple_generation',
    'layer06_concept_relation_induction',
    'layer07_hierarchisation',
    'layer08_axiom_schemata_extraction',
    'layer09_general_axiom_extraction',
    'layer10_validation_reasoning',
    'layer11_inference_completion',
    'layer12_serialization',
]
LAYER_INDEX = {name: i for i, name in enumerate(LAYER_NAMES)}


def discover_latest_layer_states(search_root: Path) -> pd.DataFrame:
    rows = []
    for p in search_root.rglob('state.json'):
        layer = p.parent.name
        if layer not in LAYER_INDEX:
            continue
        rows.append({
            'layer_index': LAYER_INDEX[layer],
            'layer_name': layer,
            'state_path': str(p),
            'mtime': p.stat().st_mtime,
            'size_mb': p.stat().st_size / (1024 * 1024),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No layer state.json files found under {search_root}')
    # Choose newest state per layer.
    df = df.sort_values(['layer_index', 'mtime'], ascending=[True, False]).groupby('layer_index', as_index=False).first()
    return df.sort_values('layer_index').reset_index(drop=True)

states_df = discover_latest_layer_states(RUNS_ROOT)
if END_STOP_INDEX is not None:
    states_df = states_df[(states_df['layer_index'] >= START_STOP_INDEX) & (states_df['layer_index'] <= END_STOP_INDEX)].copy()
else:
    states_df = states_df[states_df['layer_index'] >= START_STOP_INDEX].copy()

display(states_df[['layer_index', 'layer_name', 'state_path', 'size_mb']])
print('Stop points to run:', len(states_df))


In [ ]:

# -------------------------
# State/evidence extraction helpers
# -------------------------

def load_json(path: Path) -> Any:
    return json.loads(Path(path).read_text(encoding='utf-8'))


def clean_text(x: Any, max_chars: int | None = None) -> str:
    if x is None:
        return ''
    if isinstance(x, (dict, list)):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    s = re.sub(r'\s+', ' ', s).strip()
    if max_chars and len(s) > max_chars:
        return s[:max_chars] + '…'
    return s


def make_unit(unit_id: str, kind: str, payload: Any, max_chars: int = 1800) -> dict[str, Any]:
    return {'unit_id': unit_id, 'kind': kind, 'text': clean_text(payload, max_chars=max_chars)}


def state_to_prefix_evidence_units(state: dict[str, Any], stop_layer_name: str) -> list[dict[str, Any]]:
    '''Build compact evidence units from the cumulative state after a stop layer.

    A saved PipelineState after Layer X already contains previous layer results, so this is prefix 0..X.
    '''
    units = []
    doc = state.get('document') or {}

    # Best source for XQuality: one unit per structured alarm/message record.
    alarm_records = doc.get('alarm_records') or []
    for i, rec in enumerate(alarm_records):
        units.append(make_unit(f'alarm_record_{i:04d}', 'document.alarm_record', rec, max_chars=2400))

    # If no alarm records, fall back to table/subsection/chunks.
    if not units:
        for key in ['table_chunks', 'subsection_chunks', 'page_chunks', 'chunks']:
            chunks = doc.get(key) or []
            for i, ch in enumerate(chunks):
                units.append(make_unit(f'{key}_{i:04d}', f'document.{key}', ch, max_chars=2400))
            if units:
                break

    # Layer 4 relation assertions add relation-oriented evidence.
    for i, a in enumerate(state.get('candidate_relation_assertions') or []):
        payload = {
            'source_candidate_label': a.get('source_candidate_label'),
            'relation_label': a.get('relation_label'),
            'target_candidate_label': a.get('target_candidate_label'),
            'confidence': a.get('confidence'),
            'justification': a.get('justification'),
            'chunk_id': a.get('chunk_id'),
        }
        units.append(make_unit(f'relation_assertion_{i:04d}', 'candidate_relation_assertion', payload, max_chars=1200))

    # Layer 5+ candidate triples are direct candidate outputs.
    for i, t in enumerate(state.get('candidate_triples') or []):
        payload = {
            'subject_label': t.get('subject_label'),
            'predicate_label': t.get('predicate_label') or t.get('predicate'),
            'object_label': t.get('object_label'),
            'confidence': t.get('confidence'),
            'justification': t.get('justification'),
            'chunk_id': t.get('chunk_id'),
        }
        units.append(make_unit(f'candidate_triple_{i:04d}', 'candidate_triple', payload, max_chars=1000))

    # Concept/ontology summaries for layer 6+.
    concept_candidates = state.get('concept_candidates') or []
    if concept_candidates:
        for i, c in enumerate(concept_candidates[:400]):
            payload = {
                'label': c.get('label'),
                'concept_id': c.get('concept_id'),
                'definition': c.get('definition') or c.get('justification'),
            }
            units.append(make_unit(f'concept_{i:04d}', 'concept_candidate', payload, max_chars=600))

    ontology_relations = state.get('ontology_relation_candidates') or []
    if ontology_relations:
        for i, r in enumerate(ontology_relations[:100]):
            units.append(make_unit(f'ontology_relation_{i:04d}', 'ontology_relation_candidate', r, max_chars=600))

    # Deduplicate identical text to avoid wasting calls.
    seen = set()
    deduped = []
    for u in units:
        key = (u['kind'], u['text'])
        if key in seen:
            continue
        seen.add(key)
        deduped.append(u)
    return deduped


def chunk_batches(units: list[dict[str, Any]], batch_size: int, max_chars: int) -> list[list[dict[str, Any]]]:
    batches = []
    current = []
    current_chars = 0
    for u in units:
        u_chars = len(u.get('text', '')) + 200
        if current and (len(current) >= batch_size or current_chars + u_chars > max_chars):
            batches.append(current)
            current = []
            current_chars = 0
        current.append(u)
        current_chars += u_chars
    if current:
        batches.append(current)
    return batches


In [ ]:

# -------------------------
# Prompting and JSONL parsing
# -------------------------
ALLOWED_PREDICATES = ['TRIGGERS', 'CAUSES', 'REQUIRES', 'HANDLED_BY', 'REFERENCES']

SYSTEM_PROMPT = '''You are an information extraction finalizer for NeoOLAF/XQuality Machine32.
Return only JSONL. Each line must be one JSON object representing one triple.
Do not use markdown, code fences, comments, or explanations.

Allowed predicates and directions:
- TRIGGERS: cause/explanation -> alarm/message
- CAUSES: alarm/message -> operational effect/consequence
- REQUIRES: alarm/message -> intervention/action
- HANDLED_BY: alarm/message -> responsible actor
- REFERENCES: alarm/message -> technical reference, page, input, schema, PLC signal

Return fields exactly:
{"subject_label":"...","predicate":"TRIGGERS|CAUSES|REQUIRES|HANDLED_BY|REFERENCES","object_label":"...","confidence":0.0-1.0,"source_unit_id":"..."}

Rules:
- Use only evidence in the provided units.
- Prefer compact labels.
- Do not invent predicates.
- Do not output ontology classes as triples unless they correspond to one of the allowed predicate relations.
- If there are no triples, return an empty response.
'''


def build_user_prompt(stop_layer_name: str, batch: list[dict[str, Any]]) -> str:
    lines = [
        f'Stop-after layer: {stop_layer_name}',
        'The evidence below is the cumulative NeoOLAF prefix state available at this stop point.',
        'Generate the final extraction triples supported by this batch only.',
        '',
        'Evidence units:',
    ]
    for u in batch:
        lines.append(f"\n[unit_id={u['unit_id']} | kind={u['kind']}]\n{u['text']}")
    return '\n'.join(lines)


def strip_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r'^```(?:jsonl|json)?\s*', '', text, flags=re.I)
    text = re.sub(r'\s*```$', '', text)
    return text.strip()


def normalize_predicate(pred: str) -> str:
    pred = clean_text(pred).upper()
    pred = re.sub(r'[^A-Z0-9]+', '_', pred).strip('_')
    aliases = {'TRIGGER': 'TRIGGERS', 'CAUSE': 'CAUSES', 'REQUIRE': 'REQUIRES', 'REFERENCE': 'REFERENCES', 'RESPONSIBLE': 'HANDLED_BY'}
    return aliases.get(pred, pred)


def valid_triple_obj(obj: dict[str, Any]) -> dict[str, Any] | None:
    subj = clean_text(obj.get('subject_label') or obj.get('subject') or obj.get('head'))
    pred = normalize_predicate(obj.get('predicate') or obj.get('relation') or '')
    tail = clean_text(obj.get('object_label') or obj.get('object') or obj.get('tail'))
    if not subj or not pred or not tail or pred not in ALLOWED_PREDICATES:
        return None
    conf = obj.get('confidence', None)
    try:
        conf = float(conf)
    except Exception:
        conf = None
    return {
        'subject_label': subj,
        'predicate': pred,
        'object_label': tail,
        'confidence': conf,
        'source_unit_id': clean_text(obj.get('source_unit_id') or obj.get('unit_id')),
    }


def parse_jsonl_or_salvage(text: str) -> tuple[list[dict[str, Any]], str, int]:
    '''Parse JSONL, JSON arrays, and salvage balanced JSON objects.
    Returns triples, parse_mode, bad_lines.
    '''
    text = strip_fences(text)
    if not text:
        return [], 'empty', 0

    # Try normal JSON object/array first.
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict) and 'triples' in parsed:
            parsed = parsed['triples']
        if isinstance(parsed, dict):
            parsed = [parsed]
        if isinstance(parsed, list):
            triples = []
            for obj in parsed:
                if isinstance(obj, dict):
                    t = valid_triple_obj(obj)
                    if t:
                        triples.append(t)
            return triples, 'json', 0
    except Exception:
        pass

    triples = []
    bad = 0
    for line in text.splitlines():
        line = line.strip().strip(',')
        if not line or line.lower().startswith(('here ', 'sure', 'the triples')):
            bad += 1
            continue
        try:
            obj = json.loads(line)
            if isinstance(obj, dict):
                t = valid_triple_obj(obj)
                if t:
                    triples.append(t)
                else:
                    bad += 1
            else:
                bad += 1
        except Exception:
            bad += 1

    if triples:
        return triples, f'jsonl_salvaged_with_{bad}_bad_lines', bad

    # Balanced-object salvage from messy/truncated text.
    objs = re.findall(r'\{[^{}]*"subject_label"[^{}]*"object_label"[^{}]*\}', text, flags=re.S)
    for s in objs:
        try:
            obj = json.loads(s)
            t = valid_triple_obj(obj)
            if t:
                triples.append(t)
        except Exception:
            bad += 1
    if triples:
        return triples, f'balanced_object_salvage_with_{bad}_bad_lines', bad

    return [], f'parse_failed_with_{bad}_bad_lines', bad


def llm_call_batch(stop_layer_name: str, batch: list[dict[str, Any]], batch_id: int, out_dir: Path) -> dict[str, Any]:
    user_prompt = build_user_prompt(stop_layer_name, batch)
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            response = completion(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_prompt},
                ],
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS + attempt * 1200,
            )
            raw = response.choices[0].message.content or ''
            raw_dir = out_dir / 'raw_batches'
            raw_dir.mkdir(parents=True, exist_ok=True)
            (raw_dir / f'batch_{batch_id:04d}.txt').write_text(raw, encoding='utf-8')
            triples, mode, bad_lines = parse_jsonl_or_salvage(raw)
            return {
                'batch_id': batch_id,
                'ok': True,
                'triples': triples,
                'parse_mode': mode,
                'bad_lines': bad_lines,
                'unit_count': len(batch),
                'attempts': attempt + 1,
                'error': '',
            }
        except Exception as e:
            last_error = repr(e)
            time.sleep(RETRY_SLEEP_SECONDS * (attempt + 1))
    return {
        'batch_id': batch_id,
        'ok': False,
        'triples': [],
        'parse_mode': 'exception',
        'bad_lines': None,
        'unit_count': len(batch),
        'attempts': MAX_RETRIES + 1,
        'error': last_error or 'unknown_error',
    }


In [ ]:

# -------------------------
# Output construction helpers
# -------------------------
def deduplicate_triples(triples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen = set()
    out = []
    for t in triples:
        subj = clean_text(t.get('subject_label'))
        pred = normalize_predicate(t.get('predicate'))
        obj = clean_text(t.get('object_label'))
        if not subj or not pred or not obj:
            continue
        key = (subj.lower(), pred, obj.lower())
        if key in seen:
            continue
        seen.add(key)
        item = dict(t)
        item['triple_id'] = f'triple_{len(out):05d}'
        item['subject_label'] = subj
        item['predicate'] = pred
        item['object_label'] = obj
        out.append(item)
    return out


def build_kg(triples: list[dict[str, Any]]) -> dict[str, Any]:
    nodes = {}
    edges = []
    for t in triples:
        s = t['subject_label']
        o = t['object_label']
        p = t['predicate']
        sid = re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')[:80]
        oid = re.sub(r'[^a-z0-9]+', '_', o.lower()).strip('_')[:80]
        nodes[sid] = {'id': sid, 'label': s}
        nodes[oid] = {'id': oid, 'label': o}
        edges.append({'source': sid, 'target': oid, 'predicate': p, 'label': p, 'triple_id': t.get('triple_id')})
    return {'nodes': list(nodes.values()), 'edges': edges}


def build_ontology(triples: list[dict[str, Any]]) -> dict[str, Any]:
    concepts = sorted({t['subject_label'] for t in triples} | {t['object_label'] for t in triples}, key=str.lower)
    relations = sorted({t['predicate'] for t in triples})
    return {
        'concepts': [{'label': c} for c in concepts],
        'relations': [{'label': r} for r in relations],
        'relation_schema': {
            'TRIGGERS': 'cause/explanation -> alarm/message',
            'CAUSES': 'alarm/message -> effect/consequence',
            'REQUIRES': 'alarm/message -> intervention/action',
            'HANDLED_BY': 'alarm/message -> responsible actor',
            'REFERENCES': 'alarm/message -> technical reference',
        },
    }


def output_dir_for(idx: int, layer_name: str) -> Path:
    return OUTPUT_ROOT / f'prefix_stop_after_{idx:02d}_{layer_name}'


def is_completed(out_dir: Path) -> bool:
    return (out_dir / 'COMPLETED.json').exists() and (out_dir / 'triples.json').exists()


def mark_completed(out_dir: Path, metadata: dict[str, Any]):
    metadata = dict(metadata)
    metadata['completed_at'] = datetime.now().isoformat(timespec='seconds')
    metadata['status'] = 'completed'
    (out_dir / 'COMPLETED.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')


In [ ]:

# -------------------------
# Preview estimated calls
# -------------------------
preview_rows = []
for _, row in states_df.iterrows():
    state = load_json(Path(row['state_path']))
    units = state_to_prefix_evidence_units(state, row['layer_name'])
    batches = chunk_batches(units, BATCH_SIZE_UNITS, MAX_BATCH_CHARS)
    out_dir = output_dir_for(int(row['layer_index']), row['layer_name'])
    preview_rows.append({
        'stop_index': int(row['layer_index']),
        'layer_name': row['layer_name'],
        'units': len(units),
        'estimated_llm_calls': len(batches),
        'already_completed': is_completed(out_dir),
        'output_dir': str(out_dir),
    })
preview_df = pd.DataFrame(preview_rows)
display(preview_df)
print('Total estimated calls not already completed:', preview_df.loc[~preview_df['already_completed'], 'estimated_llm_calls'].sum())


In [ ]:

# -------------------------
# If starting at an interrupted Layer 6 folder, clean/rename it before rerun
# -------------------------
start_row = states_df[states_df['layer_index'] == START_STOP_INDEX]
if DELETE_INCOMPLETE_START_FOLDER and not start_row.empty:
    layer_name = start_row.iloc[0]['layer_name']
    out_dir = output_dir_for(START_STOP_INDEX, layer_name)
    if out_dir.exists() and not is_completed(out_dir):
        backup = out_dir.with_name(out_dir.name + '_INCOMPLETE_BACKUP_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
        print('Renaming incomplete start folder:')
        print(' ', out_dir)
        print(' ->', backup)
        shutil.move(str(out_dir), str(backup))


In [ ]:

# -------------------------
# Run/resume generation from Layer 6 onward
# -------------------------
status_rows = []
progress_log = OUTPUT_ROOT / 'resume_from_6_progress.log'

for _, row in tqdm(states_df.iterrows(), total=len(states_df), desc='Stop points'):
    idx = int(row['layer_index'])
    layer_name = row['layer_name']
    state_path = Path(row['state_path'])
    out_dir = output_dir_for(idx, layer_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    if is_completed(out_dir) and not FORCE_RERUN:
        print(f'[SKIP] already completed: {out_dir.name}')
        status_rows.append({'stop_index': idx, 'layer_name': layer_name, 'status': 'skipped_completed', 'output_dir': str(out_dir)})
        continue

    print(f'\n=== {out_dir.name} ===')
    t0 = time.time()
    state = load_json(state_path)
    units = state_to_prefix_evidence_units(state, layer_name)
    batches = chunk_batches(units, BATCH_SIZE_UNITS, MAX_BATCH_CHARS)
    print(f'Units: {len(units)} | Batches / OpenRouter calls: {len(batches)} | Workers: {MAX_WORKERS}')

    all_triples = []
    batch_meta = []
    ok = 0
    errors = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(llm_call_batch, layer_name, batch, i, out_dir): i for i, batch in enumerate(batches)}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=out_dir.name):
            result = fut.result()
            batch_meta.append({k: v for k, v in result.items() if k != 'triples'})
            all_triples.extend(result.get('triples') or [])
            if result.get('ok'):
                ok += 1
            else:
                errors += 1
                print('[ERROR]', out_dir.name, 'batch', result.get('batch_id'), result.get('error'))
            if result.get('parse_mode', '').startswith('jsonl_salvaged') and result.get('bad_lines', 0) >= 3:
                print('[SALVAGED]', out_dir.name, 'batch', result.get('batch_id'), result.get('parse_mode'))

    triples = deduplicate_triples(all_triples)
    kg = build_kg(triples)
    ontology = build_ontology(triples)

    (out_dir / 'triples.json').write_text(json.dumps(triples, indent=2, ensure_ascii=False), encoding='utf-8')
    (out_dir / 'kg.json').write_text(json.dumps(kg, indent=2, ensure_ascii=False), encoding='utf-8')
    (out_dir / 'ontology.json').write_text(json.dumps(ontology, indent=2, ensure_ascii=False), encoding='utf-8')
    pd.DataFrame(batch_meta).sort_values('batch_id').to_csv(out_dir / 'batch_metadata.csv', index=False)

    elapsed = time.time() - t0
    meta = {
        'stop_index': idx,
        'layer_name': layer_name,
        'state_path': str(state_path),
        'model': MODEL,
        'profile': PROFILE_NAME,
        'units': len(units),
        'batches': len(batches),
        'ok_batches': ok,
        'error_batches': errors,
        'triple_count': len(triples),
        'elapsed_seconds': elapsed,
        'batch_size_units': BATCH_SIZE_UNITS,
        'max_batch_chars': MAX_BATCH_CHARS,
        'max_workers': MAX_WORKERS,
    }
    (out_dir / 'metadata.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
    mark_completed(out_dir, meta)

    line = f"{datetime.now().isoformat(timespec='seconds')} {out_dir.name} triples={len(triples)} batches={len(batches)} ok={ok} errors={errors} elapsed={elapsed:.1f}s\n"
    progress_log.open('a', encoding='utf-8').write(line)
    print(line.strip())
    status_rows.append({'stop_index': idx, 'layer_name': layer_name, 'status': 'completed', **meta, 'output_dir': str(out_dir)})

status_df = pd.DataFrame(status_rows)
status_df.to_csv(OUTPUT_ROOT / 'resume_from_6_run_status.csv', index=False)
display(status_df)
print('Done. Results saved to:', OUTPUT_ROOT)


## After this finishes

Run the evaluation-only notebook:

```text
XQuality_Prefix_Stop_After_Layer_Evaluate_Completed_Only.ipynb
```

It will evaluate all completed prefix outputs and skip any incomplete folders automatically.
